✅ 4. ParentDocumentRetriever

Use Case: Retrieve parent chunks based on a retrieval of smaller child chunks, great for keeping context intact in long documents.

In [1]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.retrievers import ParentDocumentRetriever
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.storage import InMemoryStore
from langchain.schema.document import Document
from dotenv import load_dotenv
import os

# 1. Load environment
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# 1. Prepare documents
docs = [Document(page_content="LangChain helps build LLM-powered apps with memory and agents.", metadata={"id": "1"}),
        Document(page_content="Agents in LangChain use tools to answer questions.", metadata={"id": "2"})]

# here we did not pass the sample txt file rather we providd input docs in structured format

# 2. Setup child splitter
child_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

# 3. Setup vectorstore for children
embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embedding)

# 4. Parent retriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=InMemoryStore(),  # Stores parent docs
    child_splitter=child_splitter
)

# 5. Add documents
retriever.add_documents(docs)

# Why is retriever.add_documents() the "Golden Step"?
#Because it performs three actions simultaneously that you cannot do manually:

# It generates a unique ID for each Parent document.

# It pushes that ID into the metadata of every Child chunk.

# It maps that ID to the document in the InMemoryStore.

# 6. Retrieve
results = retriever.get_relevant_documents("What are agents?")
for doc in results:
    print("📄 Retrieved Doc:", doc.page_content)

C:\Users\my pc\AppData\Local\Temp\ipykernel_5328\2391120349.py:24: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding = OpenAIEmbeddings()
C:\Users\my pc\AppData\Local\Temp\ipykernel_5328\2391120349.py:38: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  results = retriever.get_relevant_documents("What are agents?")


📄 Retrieved Doc: Agents in LangChain use tools to answer questions.
📄 Retrieved Doc: LangChain helps build LLM-powered apps with memory and agents.


# Step 1: The "Indexing" Phase (Adding Documents)
When you run retriever.add_documents(docs), this happens:

Storage: The full, original documents (Parents) are saved in the InMemoryStore.

Splitting: The child_splitter breaks those Parents into many small pieces (Children).

Embedding: Those tiny Children are converted into vectors and stored in the FAISS Vector Store. Each Child is "tagged" with its Parent's ID.

# Step 2: The "Retrieval" Phase (Querying)
When you run retriever.get_relevant_documents("What are agents?"), this happens:

Query Embedding (The Correction): The retriever does NOT chunk the input query. Usually, a query is short (a single question), so it is converted into one single vector.

The Search: That query vector is compared against all the Child vectors in the FAISS store. It looks for the "Children" that are mathematically most similar to your question.

The ID Lookup: Once the best-matching Children are found, the retriever looks at their metadata to find the parent_id.

The Swap: Instead of giving you the tiny Child chunks, it goes to the InMemoryStore, pulls out the Full Parent Document associated with that ID, and returns that to you.

✅ 5. BM25Retriever

Use Case: Pure keyword-based search (like traditional search engines). No embeddings needed.

In [3]:
!pip install rank_bm25

In [4]:
from langchain.retrievers import BM25Retriever
from langchain.schema.document import Document

# Create simple text docs
docs = [
    Document(page_content="LangChain enables LLM applications."),
    Document(page_content="Vector search is powerful."),
    Document(page_content="BM25 is a classical retrieval method.")
]

# Create BM25 retriever
bm25_retriever = BM25Retriever.from_documents(docs)

# Retrieve
results = bm25_retriever.get_relevant_documents("How does BM25 work?")
for doc in results:
    print("📝 BM25 Result:", doc.page_content)

📝 BM25 Result: BM25 is a classical retrieval method.
📝 BM25 Result: Vector search is powerful.
📝 BM25 Result: LangChain enables LLM applications.


✅ 6. EnsembleRetriever

Use Case: Combine multiple retrievers (e.g., keyword + vector-based) with weighted scores.

In [5]:
from langchain.retrievers import EnsembleRetriever
from langchain.retrievers import BM25Retriever
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.schema.document import Document

# Sample docs
docs = [Document(page_content="LangChain supports LLMs."), Document(page_content="You can build AI apps using LangChain.")]

# BM25 Retriever
bm25 = BM25Retriever.from_documents(docs)

# Vector Retriever
embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embedding)
vector_retriever = vectorstore.as_retriever()

# Ensemble Retriever (equal weight)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25, vector_retriever],
    weights=[0.5, 0.5]
)

# Query
results = ensemble_retriever.get_relevant_documents("AI apps using LangChain")
for doc in results:
    print("🔍 Ensemble Doc:", doc.page_content)

🔍 Ensemble Doc: You can build AI apps using LangChain.
🔍 Ensemble Doc: LangChain supports LLMs.


7.TimeWeightedVectorStoreRetriever

This retriever boosts document relevance by factoring recency and importance of interactions (used in agents with memory or relevance ranking).

In [9]:

from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.retrievers import TimeWeightedVectorStoreRetriever
from langchain.schema import Document
from datetime import datetime
import os

# Initialize embedding & vectorstore
embedding = OpenAIEmbeddings()

# THE DOCS CAN BE ADDED USING THE RETRIEVER
docs = [
    Document(page_content="LangChain is for LLM-based apps", metadata={"last_accessed_at": datetime.now()}),
    Document(page_content="Vector search improves relevance", metadata={"last_accessed_at": datetime.now()})
]
vectorstore = FAISS.from_documents(docs, embedding)

# TimeWeighted Retriever
retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore,
    decay_rate=0.01,
    k=2,
    score_threshold=None
)

retriever.add_documents(docs) # THIS IS HOW YOU ADD DOCS

# Retrieve
results = retriever.get_relevant_documents("What is LangChain?")
for r in results:
    print(r)
    print(r.page_content)

page_content='LangChain is for LLM-based apps' metadata={'last_accessed_at': datetime.datetime(2026, 2, 25, 20, 38, 4, 138502), 'created_at': datetime.datetime(2026, 2, 25, 20, 38, 2, 831993), 'buffer_idx': 0}
LangChain is for LLM-based apps
page_content='Vector search improves relevance' metadata={'last_accessed_at': datetime.datetime(2026, 2, 25, 20, 38, 4, 138502), 'created_at': datetime.datetime(2026, 2, 25, 20, 38, 2, 831993), 'buffer_idx': 1}
Vector search improves relevance


How it contributes

It modifies the score of an item based on age.

Typical formula (exponential decay):

𝑛
𝑒
𝑤
_
𝑠
𝑐
𝑜
𝑟
𝑒
=
𝑜
𝑟
𝑖
𝑔
𝑖
𝑛
𝑎
𝑙
_
𝑠
𝑐
𝑜
𝑟
𝑒
×
𝑒
−
(
𝑑
𝑒
𝑐
𝑎
𝑦
_
𝑟
𝑎
𝑡
𝑒
×
𝑎
𝑔
𝑒
)
new_score=original_score×e
−(decay_rate×age)

What happens:

If decay rate is high

Score drops quickly

Recent items dominate

If decay rate is low

Score drops slowly

Older items remain competitive

In [10]:
results

[Document(metadata={'last_accessed_at': datetime.datetime(2026, 2, 25, 20, 38, 4, 138502), 'created_at': datetime.datetime(2026, 2, 25, 20, 38, 2, 831993), 'buffer_idx': 0}, page_content='LangChain is for LLM-based apps'),
 Document(metadata={'last_accessed_at': datetime.datetime(2026, 2, 25, 20, 38, 4, 138502), 'created_at': datetime.datetime(2026, 2, 25, 20, 38, 2, 831993), 'buffer_idx': 1}, page_content='Vector search improves relevance')]

In [8]:
docs

[Document(metadata={'last_accessed_at': datetime.datetime(2026, 2, 25, 20, 3, 9, 955058)}, page_content='LangChain is for LLM-based apps'),
 Document(metadata={'last_accessed_at': datetime.datetime(2026, 2, 25, 20, 3, 9, 955058)}, page_content='Vector search improves relevance')]

8.TavilySearchAPIRetriever


In [12]:
!pip install tavily-python

In [13]:
from langchain.retrievers import TavilySearchAPIRetriever
import os

# Set your Tavily API key
#os.environ["TAVILY_API_KEY"] = "your_tavily_api_key"

from dotenv import load_dotenv
import os
# 🔐 Load API keys
load_dotenv(".env")
openai_api_key = os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

# Initialize Tavily Retriever
retriever = TavilySearchAPIRetriever(k=3)

# Perform search
docs = retriever.get_relevant_documents("Latest updates about LangChain")
for doc in docs:
    print(doc.page_content)

New in Agent Builder: all new agent chat, file uploads + tool registry ... Read about the latest product updates, events, and content from the LangChain team.
We're kicking off 2026 with a fresh set of agent-building updates, improved experiment comparison, and new reads on observability and
New integration packages for pluggable sandboxes: langchain-modal, langchain-daytona, and langchain-runloop. See sandboxes guide and example data analysis


In [16]:
docs

[Document(metadata={'title': 'LangChain Blog', 'source': 'https://blog.langchain.com/', 'score': 0.9999535, 'images': []}, page_content='New in Agent Builder: all new agent chat, file uploads + tool registry ... Read about the latest product updates, events, and content from the LangChain team.'),
 Document(metadata={'title': 'January 2026: LangChain Newsletter', 'source': 'https://blog.langchain.com/january-2026-langchain-newsletter/', 'score': 0.9994967, 'images': []}, page_content="We're kicking off 2026 with a fresh set of agent-building updates, improved experiment comparison, and new reads on observability and"),
 Document(metadata={'title': 'Changelog - Docs by LangChain', 'source': 'https://docs.langchain.com/oss/python/releases/changelog', 'score': 0.99929583, 'images': []}, page_content='New integration packages for pluggable sandboxes: langchain-modal, langchain-daytona, and langchain-runloop. See sandboxes guide and example data analysis')]